### Listing 9.1: Merging a base model with a LoRA adapter

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_id = "mistralai/Mistral-7B-v0.1"
adapter_path = "./my-lora-adapter"
base_model = AutoModelForCausalLM.from_pretrained(
   base_model_id,
   dtype=torch.float16, # general case see warning
   device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()
model.save_pretrained("./merged-model")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained("./merged-model")


### Compiling llama.cpp on your system

In [ ]:
%%bash
if [ ! -d "llama.cpp" ]; then
    echo "Cloning llama.cpp repository..."
    git clone https://github.com/ggerganov/llama.cpp
else
    echo "Directory 'llama.cpp' already exists. Skipping clone."
fi


### Converting your merged model to GGUF

In [ ]:
%%bash
cd llama.cpp

cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j

In [ ]:
%%bash
cd llama.cpp
python convert_hf_to_gguf.py ../merged-model \
  --outfile merged-model-f16.gguf \
  --outtype f16


In [ ]:
%%bash
cd llama.cpp
./build/bin/llama-quantize merged-model-f16.gguf merged-model-Q4_K_M.gguf Q4_K_M

### Listing 9.2: Quantizing a merged model to 4-bit using AutoGPTQ

In [ ]:
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer
import torch

quantize_config = QuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False
)

model = GPTQModel.load(
    "./merged-model",
    quantize_config=quantize_config,
    torch_dtype=torch.float16,
    device="cuda:0"
)

tokenizer = AutoTokenizer.from_pretrained("./merged-model")
examples = [
    {k: v for k, v in tokenizer(text, return_tensors="pt", truncation=True, max_length=512).items()}
    for text in [
        "Quantization calibration example text",
        "The quick brown fox jumps over the lazy dog",
        "Large language models are trained on diverse datasets",
    ]
]

model.quantize(examples)
model.save_quantized("./merged-model-gptq", use_safetensors=True)

### Compressing with EXL3

In [ ]:
%%bash
if [ ! -d "exllamav3" ]; then
    echo "Cloning exllamav3 repository..."
    https://github.com/turboderp-org/exllamav3
else
    echo "Directory 'exllamav3' already exists. Skipping clone."
fi

In [ ]:
%%bash
cd exllamav3

python convert.py \
 -i ./merged-model \
 -o ./merged-model-exl3 \
 -w ./working_dir \
 -c calibration_data.parquet \
 -b 4.0

### Listing 9.3: Quantizing a merged model with AutoAWQ

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model = AutoAWQForCausalLM.from_pretrained("./merged-model")
tokenizer = AutoTokenizer.from_pretrained("./merged-model")

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

model.quantize(tokenizer, quant_config=quant_config)
model.save_quantized("./merged-model-awq")
tokenizer.save_pretrained("./merged-model-awq")

In [ ]:
%%bash
vllm serve mistralai/Mistral-7B-Instruct-v0.1 \
  --host 0.0.0.0 \
  --port 8000

curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "mistralai/Mistral-7B-Instruct-v0.1",
    "messages": [
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Explain the concept of PagedAttention."}
    ],
    "temperature": 0.7,
    "max_tokens": 150
  }'


In [ ]:
%%bash
pkill -f "vllm serve"

### Listing 9.4: Executing batch offline inference using vLLM’s Python API

In [ ]:
from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The capital of France is",
    "The future of AI serving is"
]

sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=100)

llm = LLM(model="mistralai/Mistral-7B-v0.1")

outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r} \nGenerated: {generated_text!r}\n---")


### Running vLLM with an AWQ quantyized model

In [ ]:
%%bash
vllm serve TheBloke/Mistral-7B-v0.1-AWQ \
  --quantization awq \
  --max-model-len 4096

: 

In [ ]:
%%bash
pkill -f "vllm serve"

### Running vLLM with cpu offloading

In [ ]:
%%bash
vllm serve mistralai/Mistral-7B-v0.1 \
  --cpu-offload-gb 4 \
  --gpu-memory-utilization 0.95


In [ ]:
%%bash
pkill -f "vllm serve"

### Serving a Mistral 7B model with KV cache compression with vLLM

In [ ]:
%%bash
vllm serve mistralai/Mistral-7B-v0.1 --kv-cache-dtype fp8

In [ ]:
%%bash
pkill -f "vllm serve"

### Using KV Cache Offloading with vLLM

In [ ]:
%%bash
export LMCACHE_LOCAL_CPU="True"
export LMCACHE_MAX_LOCAL_CPU_SIZE="8.0"

vllm serve mistralai/Mistral-7B-v0.1 \
  --gpu-memory-utilization 0.8 \
  --kv-transfer-config '{"kv_connector":"LMCacheConnectorV1","kv_role":"kv_both"}' 


In [ ]:
%%bash
pkill -f "vllm serve"

### Listing 9.5: Combining FP8 KV cache compression and CPU offloading in vLLM

import os 
from vllm import LLM, SamplingParams 
from vllm.config import KVTransferConfig 

os.environ["LMCACHE_LOCAL_CPU"] = "True" # enable CPU backend 
os.environ["LMCACHE_MAX_LOCAL_CPU_SIZE"] = "8.0" # limit CPU offload to 8 GB 

kv_config = KVTransferConfig(kv_connector="LMCacheConnectorV1", kv_role="kv_both") 

llm = LLM(
    model="mistralai/Mistral-7B-v0.1",
    kv_cache_dtype="fp8",
    kv_transfer_config=kv_config, 
    gpu_memory_utilization=0.8
)

### Customizing vLLM with different LoRA Adapters

In [ ]:
%%bash
vllm serve mistralai/Mistral-7B-v0.1 \
  --enable-lora \
  --lora-modules sql_coder=predibase/wikisql \
                 customer_support=predibase/customer_support

curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "sql_coder",
    "messages": [
      {"role": "system", "content": "You are a SQL expert."},
      {"role": "user", "content": "Write a SQL query to find all users whose account status is active and joined after 2023."}
    ],
    "max_tokens": 100
  }'

curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "customer_support",
    "messages": [
      {"role": "system", "content": "You are a helpful customer support agent."},
      {"role": "user", "content": "My package arrived completely crushed, I want a refund!"}
    ],
    "max_tokens": 100
  }'


In [ ]:
%%bash
pkill -f "vllm serve"

### Listing 9.6: Serving multiple LoRA adapters dynamically on a single base model

from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams 
from vllm.lora.request import LoRARequest 

sql_adapter_path = snapshot_download(repo_id="predibase/wikisql") 
support_adapter_path = snapshot_download(repo_id="predibase/customer_support")

llm = LLM(
    model="mistralai/Mistral-7B-v0.1", 
    enable_lora=True,
    max_loras=2  # Specify this if you plan to keep multiple LoRAs in memory
) 

sampling_params = SamplingParams(temperature=0.0, max_tokens=128) 

sql_prompt = "[INST] Write a SQL query to find all users whose account status is active and joined after 2023. [/INST]" 
support_prompt = "[INST] You are a helpful customer support agent. Respond to this: 'My package arrived completely crushed, I want a refund!' [/INST]"

print("--- Using SQL Adapter ---")
outputs_sql = llm.generate( 
    [sql_prompt], 
    sampling_params, 
    lora_request=LoRARequest(lora_name="sql_adapter", lora_int_id=1, lora_path=sql_adapter_path) 
) 
print(outputs_sql[0].outputs[0].text.strip()) 

print("\n--- Using Customer Support Adapter ---")
outputs_support = llm.generate( 
    [support_prompt], 
    sampling_params, 
    lora_request=LoRARequest(lora_name="support_adapter", lora_int_id=2, lora_path=support_adapter_path) 
) 
print(outputs_support[0].outputs[0].text.strip())

### Trying vLLM with FlashInfer

In [ ]:
%%bash
VLLM_ATTENTION_BACKEND=FLASHINFER 
vllm serve mistralai/Mistral-7B-v0.1

In [ ]:
%%bash
pkill -f "vllm serve"

### Falling back to running wit FlashAttention

%%bash
VLLM_ATTENTION_BACKEND=FLASH_ATTN 
vllm serve mistralai/Mistral-7B-v0.1


In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "vllm serve"])

### Continuous batching with vLLM

%%bash
vllm serve mistralai/Mistral-7B-v0.1 \
  --max-num-seqs 256 \
  --max-num-batched-tokens 8192

In [ ]:
%%bash
pkill -f "vllm serve"

### speculative decoding

%%bash
vllm serve mistralai/Mixtral-8x7B-Instruct-v0.1 \
  --speculative-model mistralai/Mistral-7B-Instruct-v0.1 \
  --num-speculative-tokens 5 \
  --tensor-parallel-size 2

In [ ]:
%%bash
pkill -f "vllm serve"

### Benchmarking with vLLM bench

In [ ]:
%%bash
vllm bench serve \
  --model mistralai/Mistral-7B-Instruct-v0.1 \
  --base-url http://localhost:8000 \
  --num-prompts 100 \
  --max-concurrency 32

### Serving with llama-cli

%%bash
cd llama.cpp
./build/bin/llama-cli \
  -m ./merged-model-Q4_K_M.gguf \
  -p "Explain the KV cache in simple terms." \
  -n 512 \
  --ctx-size 4096 \
  --threads 8 \
  --n-gpu-layers 32

# Benchmarking with llama-bench

In [ ]:
%%bash
cd llama.cpp
./llama-bench -m ./merged-model-Q4_K_M.gguf -p 512 -n 0 -b 2

In [ ]:
%%bash
cd llama.cpp
./llama-bench -m ./merged-model-Q4_K_M.gguf -p 0 -n 128

### Using llama-server

%%bash
cd llama.cpp
./build/bin/llama-server \
  -m ./merged-model-Q4_K_M.gguf \
  --ctx-size 8192 \
  --n-gpu-layers 32 \
  --port 8080

### Listing 9.7: Interacting with the local llama-server via the OpenAI Python Client

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="local-model",  # any string; llama-server ignores this field
    messages=[{"role": "user", "content": "What is the KV cache?"}]
)
print(response.choices[0].message.content)

### Ollama

%%writefile Modelfile
FROM ./merged-model-Q4_K_M.gguf

TEMPLATE """<|im_start|>system {{ .System }}<|im_end|> <|im_start|>user {{ .Prompt }}<|im_end|> <|im_start|>assistant """

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096

SYSTEM "You are a specialized assistant for medical triage. Answer concisely and always recommend consulting a physician for diagnosis."

In [ ]:
%%bash
ollama create my-custom-slm -f Modelfile
ollama run my-custom-slm

### Listing 9.8: Streaming inference with the Ollama Python library

In [ ]:
import ollama

for chunk in ollama.chat(
    model="my-custom-slm",
    messages=[
        {
            "role": "user",
            "content": "Patient presents with a fever of 39°C and a sore throat for two days. What are the recommended next steps?"
        }
    ],
    stream=True
):
    print(chunk["message"]["content"], end="", flush=True)


### Listing 9.9: Using Ollama through the OpenAI-compatible endpoint

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"          # required by the SDK, ignored by Ollama
)

response = client.chat.completions.create(
    model="my-custom-slm",
    messages=[
        {
            "role": "user",
            "content": "Patient presents with a fever of 39°C and a sore throat for two days. What are the recommended next steps?"
        }
    ]
)

print(response.choices[0].message.content)